# Demo BTL01 – Xấp xỉ hàm Gaussian 2D bằng MLP (PyTorch)
Notebook minh họa nhanh, dùng lại các module trong `src/`. Chạy từ thư mục gốc của dự án (hoặc từ thư mục chứa notebook này).
Toàn bộ thí nghiệm A/B/C chạy bằng `python src/run_experiments.py` (xem README).

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) in ("notebooks", "report", "src"):
    os.chdir("..")
sys.path.insert(0, "src")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from data import load_all, gaussian_2d, RANGE_X1, RANGE_X2
from model import MLP
from train import fit, evaluate_all, predict, Scaler, ACTS
print("PyTorch", torch.__version__)

## 1. Dữ liệu (train pool 2000 / validation 200 / test 100, seed 42 / 1042 / 2042)

In [ ]:
data = load_all("data")
for k, (x, y) in data.items():
    print(f"{k:5s} x{x.shape} y{y.shape}  y in [{y.min():.2e}, {y.max():.4f}]")
fig = plt.figure(figsize=(12, 3.6))
for i, (k, c) in enumerate(zip(["train", "val", "test"], ["tab:green", "tab:red", "tab:blue"])):
    ax = fig.add_subplot(1, 3, i + 1, projection="3d"); x, y = data[k]
    ax.scatter(x[:, 0], x[:, 1], y, c=c, s=4); ax.set_title(k)
plt.tight_layout(); plt.show()

## 2. Mô hình MLP (`nn.Module`) – ví dụ 1 tầng ẩn, 10 nơ-ron

In [ ]:
model = MLP((10,), ACTS["tanh"])
print(model)
print("Số tham số:", sum(p.numel() for p in model.parameters()))

## 3. Huấn luyện nhanh (A = 2000, B = 10, C = 1, 100 epoch)
Vòng lặp tự viết: forward → loss → zero_grad → backward → step; checkpoint theo validation loss.

In [ ]:
xtr, ytr = data["train"][0][:2000], data["train"][1][:2000]
model, sc, hist, info = fit((10,), xtr, ytr, *data["val"], seed=0, cfg=dict(epochs=100))
h = pd.DataFrame(hist, columns=["epoch", "train_loss", "val_loss"])
plt.figure(figsize=(5.5, 3.5)); plt.plot(h.epoch, h.train_loss, label="train"); plt.plot(h.epoch, h.val_loss, label="validation")
plt.yscale("log"); plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.legend(); plt.grid(alpha=.3); plt.show()
print(info)

## 4. Đánh giá (test chỉ dùng để đánh giá cuối)

In [ ]:
res = evaluate_all(model, sc, data)
pd.DataFrame(res).T

## 5. y thật vs ŷ dự đoán và bề mặt xấp xỉ

In [ ]:
gx1, gx2 = np.meshgrid(np.linspace(*RANGE_X1, 60), np.linspace(*RANGE_X2, 60))
G = np.stack([gx1.ravel(), gx2.ravel()], 1)
Z_true = gaussian_2d(G).reshape(gx1.shape)
Z_hat = predict(model, sc(G.astype(np.float32))).reshape(gx1.shape)
xt, yt = data["test"]; yh = predict(model, sc(xt))
fig = plt.figure(figsize=(13, 3.8))
ax = fig.add_subplot(131); ax.scatter(yt, yh, s=8); lim = [0, yt.max() * 1.05]; ax.plot(lim, lim, "k--"); ax.set_xlabel("y thật"); ax.set_ylabel("ŷ"); ax.set_title("Test")
ax = fig.add_subplot(132, projection="3d"); ax.plot_surface(gx1, gx2, Z_true, cmap="viridis"); ax.set_title("Bề mặt thật")
ax = fig.add_subplot(133, projection="3d"); ax.plot_surface(gx1, gx2, Z_hat, cmap="plasma"); ax.set_title("Bề mặt xấp xỉ")
plt.tight_layout(); plt.show()

## 6. Nạp mô hình cuối đã lưu (`checkpoints/best_model.pt`)

In [ ]:
ck = torch.load("checkpoints/best_model.pt", weights_only=False)
m = MLP(tuple(ck["hidden"]), ACTS[ck["cfg"]["activation"]]); m.load_state_dict(ck["state_dict"])
sc2 = Scaler(xtr); sc2.m, sc2.s = ck["x_mean"], ck["x_std"]
print("Kiến trúc:", ck["hidden"])
pd.DataFrame(evaluate_all(m, sc2, data)).T